# 02. Reproduce LightGBM and tune PPG-only alternatives

This notebook is a thin, reproducible research driver around the project scripts.
It avoids embedding private paths or bulky outputs; generated artifacts are written
under ignored `data/processed/` and `artifacts/` directories.

Primary target: self-reported discrete emotion (`EmotType`, exposed as `reported`).
Default split: participant-grouped cross-validation to avoid subject leakage.

In [ ]:
from pathlib import Path
import json
import subprocess

ROOT = Path.cwd()
DATA_ROOT = ROOT / "data" / "extracted"
FEATURE_ALL = ROOT / "data" / "processed" / "features_all.parquet"
FEATURE_PPG = ROOT / "data" / "processed" / "features_ppg.parquet"
RESULTS = ROOT / "artifacts" / "results"
OPTUNA = ROOT / "artifacts" / "optuna"

print(ROOT)
print("extracted data exists:", DATA_ROOT.exists())

## 1. Prepare data

Run this once in a fresh clone. It downloads Figshare v6 files, verifies checksums,
extracts participant archives, and writes a dataset index.

In [ ]:
# subprocess.run(["uv", "run", "python", "scripts/download_llamac.py", "--prepare"], check=True)

## 2. Build feature tables

The all-channel table mirrors the official Figshare notebook feature families
(EEG, GSR, PPG, SKT, RESP). The PPG table keeps only PPG-derived columns.

In [ ]:
# All-channel features can take several minutes because every EEG CSV is summarized.
# subprocess.run([
#     "uv", "run", "--group", "ml", "python", "scripts/build_features.py",
#     "--mode", "all", "--workers", "4", "--output", str(FEATURE_ALL),
# ], check=True)

subprocess.run([
    "uv", "run", "--group", "ml", "python", "scripts/build_features.py",
    "--mode", "ppg", "--workers", "4", "--output", str(FEATURE_PPG),
], check=True)

## 3. Evaluate LightGBM baselines

These commands produce JSON files with top-1/top-2/top-3 accuracy, macro F1,
weighted F1, balanced accuracy, Cohen's kappa, and confusion matrices.

In [ ]:
# Source-notebook-style all-channel LightGBM with leakage-safe grouped split.
# subprocess.run([
#     "uv", "run", "--group", "ml", "python", "scripts/train_model.py",
#     "--features", str(FEATURE_ALL), "--model", "lightgbm",
#     "--feature-set", "all", "--target", "reported", "--split", "grouped",
#     "--folds", "5", "--device", "auto", "--output-dir", str(RESULTS),
# ], check=True)

# PPG-only LightGBM baseline.
subprocess.run([
    "uv", "run", "--group", "ml", "python", "scripts/train_model.py",
    "--features", str(FEATURE_PPG), "--model", "lightgbm",
    "--feature-set", "ppg", "--target", "reported", "--split", "grouped",
    "--folds", "5", "--device", "auto", "--output-dir", str(RESULTS),
], check=True)

## 4. Optuna alternatives

Tune several non-DNN families against the same grouped split and primary metric.
Increase `--trials` for serious runs; keep final test/held-out folds outside the
search loop for publication-quality claims.

In [ ]:
subprocess.run([
    "uv", "run", "--group", "ml", "python", "scripts/tune_models.py",
    "--features", str(FEATURE_PPG), "--feature-set", "ppg", "--target", "reported",
    "--split", "grouped", "--folds", "5",
    "--models", "lightgbm", "extra_trees", "hist_gradient_boosting",
    "--trials", "20", "--metric", "macro_f1", "--device", "auto",
    "--output-dir", str(OPTUNA),
], check=True)

## 5. Summarize comparable results

In [ ]:
subprocess.run([
    "uv", "run", "python", "scripts/summarize_results.py",
    str(RESULTS), str(OPTUNA / "locked-results"),
    "--output", str(RESULTS / "summary.csv"),
], check=True)

print((RESULTS / "summary.csv").read_text()[:2000])